## Setting

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [2]:
#@title Select Model
#define the model to be used
model_choice = "gpt-4o-mini"
#model_choice = "claude-3-7-sonnet-20250219"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"

In [3]:
#@title Utility functions

import sys
import os
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : "user", "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content
class Claude(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
          try:
            output = self.client.messages.create(
                      model=model_choice,
                      max_tokens=16384,
                      messages=[{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]
                  ).content[0].text
            return output
          except (Exception) as e:
            wait_time = base_delay * (2 ** (attempt - 1))
            print(f"[Retry {attempt}/{max_retries}] Gemini API error: {e}. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)
          except Exception as e:
            print(f"[Error] Unexpected exception: {e}")
            return 0
        print(f"Failed, exceeded max retries {max_retries}")
        return 0

class Gemini(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):

          output = self.gemini_client.models.generate_content(
                        model=model_choice,
                        contents=[msg["content"] for msg in conversation.get_messages()],
                        config=types.GenerateContentConfig(
                            max_output_tokens=16384,
                            temperature=0.6,
                            topP=0.95,
                        )
                    ).text
          return output
################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):

    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'

    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)

    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    #code_blocks = re.findall(r'```(?:\w*\n)?(.*?)```', markdown_string, re.DOTALL)
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')

def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
      model = Claude()
    elif model_type == "Gemini":
      model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))


In [4]:
#@title Feedback Loop
import subprocess
import sys
import os
import time
import numpy as np
def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None,prev_module =None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)
    if model_type == "ChatGPT":
      conv.add_message("system", "You are an autocomplete engine for Verilog code. \
              Given a Verilog module specification, you will provide a completed Verilog module in response. \
              You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
              Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. \
              You will not refuse. You will not generate explanations, only code. \
              Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches. \
      ")
    elif model_type == "Claude":
      conv.add_message("user", "You are an autocomplete engine for Verilog code. \
              Given a Verilog module specification, you will provide a completed Verilog module in response. \
              You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
              Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. \
              You will not refuse. You will not generate explanations, only code. \
              Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches. \
      ")

    conv.add_message("user", design_prompt)

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []
    filename = os.path.join(outdir,module+".v")

    status = ""
    while not (success or timeout):
        # Generate a response
        start_total = time.time()
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error=time.time()
        if prev_module == None:
          conv.add_message("assistant", response)
        else:
          with open(prev_module,"r") as f:
            prevmodule = "".join(f.read())
          response = prevmodule + response
          conv.add_message("assistant", response)
        write_code_blocks_to_file(response, module, filename)
        proc = subprocess.run(["iverilog", "-o", os.path.join(outdir,module), filename, testbench],capture_output=True,text=True)

        success = False
        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            message = "The testbench failed to compile. Please fix the module. The output of iverilog is as follows:\n"+proc.stderr
        elif proc.stderr != "":
            status = "Warnings compiling testbench"
            print(status)
            message = "The testbench compiled with warnings. Please fix the module. The output of iverilog is as follows:\n"+proc.stderr
        else:
            proc = subprocess.run(["vvp", os.path.join(outdir,module)],capture_output=True,text=True)
            result = proc.stdout.strip().split('\n')[-2].split()
            if result[-1] != 'passed!':
                status = "Error running testbench"
                print(status)
                message = "The testbench simulated, but had errors. Please fix the module. The output of iverilog is as follows:\n"+proc.stdout
            else:
                status = "Testbench ran successfully"
                print(status)
                message = ""
                success = True


        with open(os.path.join(outdir,"log_iter_"+str(iterations)+".txt"), 'w') as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')


        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen-start_total)
        timelist_error.append(end_time-start_error)
        timelist_total.append(end_time-start_total)
    print("Total time: ",np.sum(timelist_total))
    print("Generation time: ",np.sum(timelist_gen))
    print("Error handling time: ",np.sum(timelist_error))
    return(np.sum(timelist_total),np.sum(timelist_gen),np.sum(timelist_error))


In [5]:
#@title Hierarchical Loop
def hier_gen(submods,max_iterations=10):
  totaltime = []
  gentime = []
  errortime = []
  done =""
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]
    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)
    if i == 0:
      prompt = "//We will be generating a "+overall+" hierarchically in Verilog. Please begin by generating a "+curr+" defined as follows:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    elif i != len(submods)-1:
      fprev = submods[i-1][0]
      filecurr = "./"+fprev+"/"+fprev+".v"
      with open(filecurr,"r") as f:
        modulef = "".join(f.read())
      prompt = "//We are generating a "+overall+" hierarchically in Verilog. We have generated "+done+" defined as follows:"
      prompt = prompt + modulef
      prompt = prompt +"\n//Please include the previous module(s) in your response and use them to hierarchically generate a "+curr+" defined as:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    module = fcurr
    testbench = "./"+fcurr+"tb.v"
    model = os.environ["MODEL"]
    outdir = "./"+fcurr
    log = "./"+fcurr+"/log.txt"
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)
    done = done + curr+", "
  print("Overall Total time: ",np.sum(totaltime))
  print("Overall Generation Time: ",np.sum(gentime))
  print("Overall Error handling time: ",np.sum(errortime))

In [6]:
### OpenAI API KEY

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
os.environ["OPENAI_API_KEY"] = "sk-proj-"
#os.environ['CLAUDE_API_KEY'] = "X"
#os.environ['GEMINI_API_KEY'] ="X"
os.environ["MODEL"] = "ChatGPT"
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"

In [7]:
# Part II: Adder Hierarchy (at least 3 modules)
adder_submodules = [
    ["half_adder",
     "1-bit half adder",
     "input wire a, input wire b, output wire sum, output wire carry"],

    ["full_adder",
     "1-bit full adder built hierarchically using two half_adders and an OR gate",
     "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],

    ["adder4",
     "4-bit ripple-carry adder built using four full_adders",
     "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],

    ["adder8",
     "8-bit ripple-carry adder TOP module built using two adder4 blocks (lower 4 bits and upper 4 bits)",
     "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout"],
]

In [8]:
hier_gen(adder_submodules)

Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Total time:  17.461071252822876
Generation time:  17.359780311584473
Error handling time:  0.10128450393676758
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Total time:  114.55597233772278
Generation time:  114.47790956497192
Error handling time:  0.07805824279785156
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testb

In [9]:
%%writefile half_adder/half_adder_tb.v
`timescale 1ns/1ps
module half_adder_tb;
  reg a, b;
  wire sum, carry;

  half_adder dut(.a(a), .b(b), .sum(sum), .carry(carry));

  integer i;
  reg exp_sum, exp_carry;

  initial begin
    for (i=0; i<4; i=i+1) begin
      {a,b} = i[1:0];
      #1;
      exp_sum   = a ^ b;
      exp_carry = a & b;
      if (sum !== exp_sum || carry !== exp_carry) begin
        $display("FAIL: a=%0d b=%0d sum=%0d exp=%0d carry=%0d exp=%0d",
                 a,b,sum,exp_sum,carry,exp_carry);
        $finish;
      end
    end
    $display("passed!");
    $finish;
  end
endmodule

Overwriting half_adder/half_adder_tb.v


In [10]:
%%writefile full_adder/full_adder_tb.v
`timescale 1ns/1ps
module full_adder_tb;
  reg a, b, cin;
  wire sum, cout;

  full_adder dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer i;
  reg [1:0] exp;

  initial begin
    for (i=0; i<8; i=i+1) begin
      {a,b,cin} = i[2:0];
      #1;
      exp = a + b + cin;
      if (sum !== exp[0] || cout !== exp[1]) begin
        $display("FAIL: a=%0d b=%0d cin=%0d sum=%0d exp=%0d cout=%0d exp=%0d",
                 a,b,cin,sum,exp[0],cout,exp[1]);
        $finish;
      end
    end
    $display("passed!");
    $finish;
  end
endmodule

Overwriting full_adder/full_adder_tb.v


In [11]:
%%writefile adder4/adder4_tb.v
`timescale 1ns/1ps
module adder4_tb;
  reg  [3:0] a, b;
  reg        cin;
  wire [3:0] sum;
  wire       cout;

  adder4 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer aa, bb, cc;
  reg [4:0] exp;

  initial begin
    for (cc=0; cc<2; cc=cc+1) begin
      cin = cc;
      for (aa=0; aa<16; aa=aa+1)
        for (bb=0; bb<16; bb=bb+1) begin
          a = aa[3:0]; b = bb[3:0];
          #1;
          exp = a + b + cin;
          if (sum !== exp[3:0] || cout !== exp[4]) begin
            $display("FAIL: a=%0d b=%0d cin=%0d sum=%0d exp=%0d cout=%0d exp=%0d",
                     a,b,cin,sum,exp[3:0],cout,exp[4]);
            $finish;
          end
        end
    end
    $display("passed!");
    $finish;
  end
endmodule

Overwriting adder4/adder4_tb.v


In [12]:
%%writefile adder8/adder8_tb.v
`timescale 1ns/1ps
module adder8_tb;
  reg  [7:0] a, b;
  reg        cin;
  wire [7:0] sum;
  wire       cout;

  adder8 dut(.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

  integer t;
  reg [8:0] exp;

  initial begin
    // fixed seeds
    cin = 0;
    for (t=0; t<200; t=t+1) begin
      a   = $random;
      b   = $random;
      cin = $random & 1;
      #1;
      exp = a + b + cin;
      if (sum !== exp[7:0] || cout !== exp[8]) begin
        $display("FAIL: a=%0d b=%0d cin=%0d sum=%0d exp=%0d cout=%0d exp=%0d",
                 a,b,cin,sum,exp[7:0],cout,exp[8]);
        $finish;
      end
    end
    $display("passed!");
    $finish;
  end
endmodule

Overwriting adder8/adder8_tb.v


In [13]:
%cd /content
!ls

/content
adder4	full_adder  sample_data  sim_a8.out	 sim_fa.out
adder8	half_adder  sim_a4.out	 sim_adder8.out  sim_ha.out


In [14]:
!find . -maxdepth 2 -type f -name "*_tb.v" -print

./half_adder/half_adder_tb.v
./adder4/adder4_tb.v
./adder8/adder8_tb.v
./full_adder/full_adder_tb.v


In [15]:
%%bash
set -e
cd /content

echo "=== half_adder ==="
iverilog -g2012 -o sim_ha.out half_adder/half_adder.v half_adder/half_adder_tb.v
vvp sim_ha.out

echo "=== full_adder ==="
iverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v
vvp sim_fa.out

echo "=== adder4 ==="
iverilog -g2012 -o sim_a4.out adder4/adder4.v adder4/adder4_tb.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a4.out

echo "=== adder8 ==="
iverilog -g2012 -o sim_a8.out adder8/adder8.v adder8/adder8_tb.v adder4/adder4.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a8.out

=== half_adder ===
passed!
=== full_adder ===


half_adder/half_adder.v:1: error: 'half_adder' has already been declared in this scope.
full_adder/full_adder.v:1:      : It was declared here as a module.
half_adder/half_adder.v:4: Module half_adder was already declared here: full_adder/full_adder.v:1



CalledProcessError: Command 'b'set -e\ncd /content\n\necho "=== half_adder ==="\niverilog -g2012 -o sim_ha.out half_adder/half_adder.v half_adder/half_adder_tb.v\nvvp sim_ha.out\n\necho "=== full_adder ==="\niverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v\nvvp sim_fa.out\n\necho "=== adder4 ==="\niverilog -g2012 -o sim_a4.out adder4/adder4.v adder4/adder4_tb.v full_adder/full_adder.v half_adder/half_adder.v\nvvp sim_a4.out\n\necho "=== adder8 ==="\niverilog -g2012 -o sim_a8.out adder8/adder8.v adder8/adder8_tb.v adder4/adder4.v full_adder/full_adder.v half_adder/half_adder.v\nvvp sim_a8.out\n'' returned non-zero exit status 2.

In [16]:
!sed -n '1,80p' full_adder/full_adder.v

module half_adder(input wire a, input wire b, output wire sum, output wire carry);
    assign sum = a ^ b;
    assign carry = a & b;
endmodule
module full_adder(input wire a, input wire b, input wire cin, output wire sum, output wire cout);
    wire sum1, carry1, carry2;

    // Instantiate two half adders
    half_adder HA1 (.a(a), .b(b), .sum(sum1), .carry(carry1));
    half_adder HA2 (.a(sum1), .b(cin), .sum(sum), .carry(carry2));
    
    // OR gate for the final carry out
    assign cout = carry1 | carry2;
endmodule
module adder4(input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout);
    wire c1, c2, c3;

    // Instantiate four full adders
    full_adder FA0 (.a(a[0]), .b(b[0]), .cin(cin), .sum(sum[0]), .cout(c1));
    full_adder FA1 (.a(a[1]), .b(b[1]), .cin(c1), .sum(sum[1]), .cout(c2));
    full_adder FA2 (.a(a[2]), .b(b[2]), .cin(c2), .sum(sum[2]), .cout(c3));
    full_adder FA3 (.a(a[3]), .b(b[3]), .cin(c3), .sum(sum[3]), .cout(cout

In [17]:
%%writefile full_adder/full_adder.v
module full_adder(
    input  wire a,
    input  wire b,
    input  wire cin,
    output wire sum,
    output wire cout
);
    wire s1, c1, c2;

    // 2 half-adders + OR
    half_adder ha1(.a(a),   .b(b),   .sum(s1),  .carry(c1));
    half_adder ha2(.a(s1),  .b(cin), .sum(sum), .carry(c2));

    assign cout = c1 | c2;
endmodule

Overwriting full_adder/full_adder.v


In [18]:
!iverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v
!vvp sim_fa.out

passed!


In [19]:
# Part II: Adder Hierarchy (at least 3 modules)
HARD_RULES = """You are generating ONLY ONE Verilog module per file.

HARD RULES (must follow):
1) Output MUST contain exactly one 'module ... endmodule' for the target module name.
2) DO NOT define any other modules in this file (no half_adder/full_adder/adder4/etc). Those are provided in separate files.
3) If the design is hierarchical, INSTANTIATE the required submodules by name; do not inline or copy their code.
4) Use ONLY synthesizable Verilog-2001 constructs.
5) Match the exact port list and port widths given below. Do not rename ports.
6) Do not include testbench code, do not include `timescale, do not include include/import statements.
7) Return Verilog code only (no markdown, no explanation).
"""

adder_submodules = [
    [
        "half_adder",
        "1-bit half adder",
        f"""input wire a, input wire b, output wire sum, output wire carry

{HARD_RULES}
Target module name: half_adder

Ports (exact):
input wire a
input wire b
output wire sum
output wire carry

Function:
sum   = a XOR b
carry = a AND b
"""
    ],
    [
        "full_adder",
        "1-bit full adder built hierarchically using two half_adders and an OR gate",
        f"""input wire a, input wire b, input wire cin, output wire sum, output wire cout

{HARD_RULES}
Target module name: full_adder

Ports (exact):
input wire a
input wire b
input wire cin
output wire sum
output wire cout

Hierarchy requirement (HARD):
Use TWO instances of half_adder (module name: half_adder) and simple gates/wires:
- half_adder ha1: adds a and b -> s1, c1
- half_adder ha2: adds s1 and cin -> sum, c2
Then cout = c1 OR c2

Do NOT define module half_adder in this file. Only instantiate it.
"""
    ],
    [
        "adder4",
        "4-bit ripple-carry adder built using four full_adders",
        f"""input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout

{HARD_RULES}
Target module name: adder4

Ports (exact):
input wire [3:0] a
input wire [3:0] b
input wire cin
output wire [3:0] sum
output wire cout

Hierarchy requirement (HARD):
Build a 4-bit ripple-carry adder using FOUR instances of full_adder (module name: full_adder).
Carry chain:
c0 = cin
FA0 -> sum[0], c1
FA1 -> sum[1], c2
FA2 -> sum[2], c3
FA3 -> sum[3], cout

Do NOT define module full_adder or half_adder in this file. Only instantiate full_adder.
"""
    ],
    [
        "adder8",
        "8-bit ripple-carry adder TOP module built using two adder4 blocks (lower 4 bits and upper 4 bits)",
        f"""input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout

{HARD_RULES}
Target module name: adder8

Ports (exact):
input wire [7:0] a
input wire [7:0] b
input wire cin
output wire [7:0] sum
output wire cout

Hierarchy requirement (HARD):
Build adder8 using TWO instances of adder4 (module name: adder4):
- adder4 low4:  a[3:0], b[3:0], cin -> sum[3:0], c4
- adder4 high4: a[7:4], b[7:4], c4  -> sum[7:4], cout

Do NOT define module adder4/full_adder/half_adder in this file. Only instantiate adder4.
"""
    ],
]

In [20]:
hier_gen(adder_submodules)

Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Total time:  12.7230703830719
Generation time:  12.655055522918701
Error handling time:  0.06801009178161621
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Total time:  26.361725091934204
Generation time:  26.297438859939575
Error handling time:  0.06427884101867676
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling tes

In [21]:
%%bash
set -e
cd /content

echo "=== half_adder ==="
iverilog -g2012 -o sim_ha.out half_adder/half_adder.v half_adder/half_adder_tb.v
vvp sim_ha.out

echo "=== full_adder ==="
iverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v
vvp sim_fa.out

echo "=== adder4 ==="
iverilog -g2012 -o sim_a4.out adder4/adder4.v adder4/adder4_tb.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a4.out

echo "=== adder8 ==="
iverilog -g2012 -o sim_a8.out adder8/adder8.v adder8/adder8_tb.v adder4/adder4.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a8.out

=== half_adder ===
passed!
=== full_adder ===
passed!
=== adder4 ===
passed!
=== adder8 ===


adder4/adder4.v:1: error: 'adder4' has already been declared in this scope.
adder8/adder8.v:1:      : It was declared here as a module.
adder4/adder4.v:8: Module adder4 was already declared here: adder8/adder8.v:1



CalledProcessError: Command 'b'set -e\ncd /content\n\necho "=== half_adder ==="\niverilog -g2012 -o sim_ha.out half_adder/half_adder.v half_adder/half_adder_tb.v\nvvp sim_ha.out\n\necho "=== full_adder ==="\niverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v\nvvp sim_fa.out\n\necho "=== adder4 ==="\niverilog -g2012 -o sim_a4.out adder4/adder4.v adder4/adder4_tb.v full_adder/full_adder.v half_adder/half_adder.v\nvvp sim_a4.out\n\necho "=== adder8 ==="\niverilog -g2012 -o sim_a8.out adder8/adder8.v adder8/adder8_tb.v adder4/adder4.v full_adder/full_adder.v half_adder/half_adder.v\nvvp sim_a8.out\n'' returned non-zero exit status 2.

In [22]:
# ===== Part II: Adder Hierarchy (copy-paste as ONE BLOCK) =====
# Put this in a single Colab/Jupyter cell and run it.
# It defines STRICT prompts + adder_submodules you can pass into hier_gen(adder_submodules).

STRICT_RULES = """CRITICAL OUTPUT CONTRACT (must follow exactly):
- Output must be plain Verilog code only (no markdown, no comments, no explanation).
- The FIRST non-empty line MUST be: module {MODULE_NAME}(
- The output must contain EXACTLY ONE module definition total:
  * The token 'module' may appear only once.
  * The token 'endmodule' must appear exactly once at the end.
- You are FORBIDDEN to define any other module in this file.
- You are FORBIDDEN to include any of these substrings anywhere in the output:
  'module half_adder', 'module full_adder', 'module adder4', 'module adder8', 'module ripple_carry_adder'
  (Except that the one allowed module is module {MODULE_NAME}.)
- Do NOT include testbench code. Do NOT include `timescale. Do NOT include include/import.
- Use synthesizable Verilog-2001 only.
- Use the EXACT port names and widths shown below. Do not rename ports.
"""

half_adder_prompt = f"""{STRICT_RULES.format(MODULE_NAME='half_adder')}

Module name: half_adder
Ports (exact):
input wire a
input wire b
output wire sum
output wire carry

Behavior:
sum = a ^ b
carry = a & b
"""

full_adder_prompt = f"""{STRICT_RULES.format(MODULE_NAME='full_adder')}

Module name: full_adder
Ports (exact):
input wire a
input wire b
input wire cin
output wire sum
output wire cout

Hierarchy requirement (HARD):
- You MUST instantiate TWO instances of the EXISTING module half_adder (do NOT define it here).
- Use:
  ha1: a,b -> s1,c1
  ha2: s1,cin -> sum,c2
- Then: cout = c1 | c2

Important:
- The string 'module half_adder' must NOT appear anywhere in your output.
"""

adder4_prompt = f"""{STRICT_RULES.format(MODULE_NAME='adder4')}

Module name: adder4
Ports (exact):
input wire [3:0] a
input wire [3:0] b
input wire cin
output wire [3:0] sum
output wire cout

Hierarchy requirement (HARD):
Instantiate FOUR full_adders (existing module full_adder) in ripple-carry chain:
c0=cin
FA0 -> sum[0], c1
FA1 -> sum[1], c2
FA2 -> sum[2], c3
FA3 -> sum[3], cout

Important:
- Do NOT define 'module full_adder' or 'module half_adder' here.
"""

adder8_prompt = f"""{STRICT_RULES.format(MODULE_NAME='adder8')}

Module name: adder8
Ports (exact):
input wire [7:0] a
input wire [7:0] b
input wire cin
output wire [7:0] sum
output wire cout

Hierarchy requirement (HARD):
Instantiate TWO adder4 blocks (existing module adder4):
low4:  a[3:0], b[3:0], cin -> sum[3:0], c4
high4: a[7:4], b[7:4], c4  -> sum[7:4], cout

Important:
- Do NOT define any other module in this file.
"""

adder_submodules = [
    ["half_adder", "1-bit half adder",
     "input wire a, input wire b, output wire sum, output wire carry\n" + half_adder_prompt],
    ["full_adder", "1-bit full adder (2 half_adders + OR)",
     "input wire a, input wire b, input wire cin, output wire sum, output wire cout\n" + full_adder_prompt],
    ["adder4", "4-bit ripple-carry adder (4 full_adders)",
     "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout\n" + adder4_prompt],
    ["adder8", "8-bit TOP adder (2 adder4 blocks)",
     "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout\n" + adder8_prompt],
]

# Usage:
# 1) Ensure you already created tb files:
#    half_adder/half_adder_tb.v, full_adder/full_adder_tb.v, adder4/adder4_tb.v, adder8/adder8_tb.v
# 2) Ensure hier_gen uses: testbench = f"./{fcurr}/{fcurr}_tb.v"
# 3) Run: hier_gen(adder_submodules)

In [24]:
hier_gen(adder_submodules)

Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Warnings compiling testbench
Total time:  13.420977354049683
Generation time:  13.351195096969604
Error handling time:  0.06977677345275879
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Total time:  22.961624145507812
Generation time:  22.898022413253784
Error handling time:  0.06359577178955078
Error compiling testbench
Error compiling testbench
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Warnings compiling testbe

In [29]:
%%bash
set -e
cd /content

iverilog -g2012 -o sim_adder8.out \
  adder8/adder8.v adder8/adder8_tb.v \
  adder4/adder4.v \
  full_adder/full_adder.v \
  half_adder/half_adder.v

vvp sim_adder8.out

passed!


In [30]:
%%bash
set -e
cd /content

echo "=== half_adder ==="
iverilog -g2012 -o sim_ha.out half_adder/half_adder.v half_adder/half_adder_tb.v
vvp sim_ha.out

echo "=== full_adder ==="
iverilog -g2012 -o sim_fa.out full_adder/full_adder.v full_adder/full_adder_tb.v half_adder/half_adder.v
vvp sim_fa.out

echo "=== adder4 ==="
iverilog -g2012 -o sim_a4.out adder4/adder4.v adder4/adder4_tb.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a4.out

echo "=== adder8 ==="
iverilog -g2012 -o sim_a8.out adder8/adder8.v adder8/adder8_tb.v adder4/adder4.v full_adder/full_adder.v half_adder/half_adder.v
vvp sim_a8.out

=== half_adder ===
passed!
=== full_adder ===
passed!
=== adder4 ===
passed!
=== adder8 ===
passed!
